In [ ]:
# ============================================
# DEEPFAKE DETECTION - KAGGLE PREPROCESSING
# ============================================
# Preprocesses raw video datasets into face crops.
# Supports any num_frames (T=16, T=32, etc.).
#
# USAGE:
# 1. Add raw video dataset as Kaggle input (must have real/ and fake/ dirs)
# 2. Set NUM_FRAMES below (default=32)
# 3. Set DATASET_NAME (e.g., 'DFDC02', 'DFD01')
# 4. Run the notebook
# 5. Download preprocessed_{DATASET_NAME}_{NUM_FRAMES}.tar.gz
#
# PREREQUISITES:
# - Raw video dataset with structure:
#   input_root/real/*.mp4  and  input_root/fake/*.mp4
#   (or nested: input_root/real/subdir/*.mp4)

# =============================================
# CONFIGURATION - CHANGE THESE!
# =============================================
NUM_FRAMES = 32       # Number of frames per video (T)
DATASET_NAME = 'DFDC02'  # Dataset identifier for output naming
OUTPUT_SIZE = 224     # Face crop size
MIN_FACE_CONF = 0.90  # MTCNN confidence threshold
MIN_DETECTION_RATIO = 0.55

import subprocess, sys, os

# Step 1: Install dependencies
print("=== Installing dependencies ===")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2", "scipy<1.17", "scikit-learn", "timm", "facenet-pytorch",
                "opencv-python-headless", "tqdm"],
               check=True)
print("Dependencies installed.")

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(["rm", "-rf", "/kaggle/working/project"], check=False)
    subprocess.run(["git", "clone", "https://github.com/Jokeera/deepfake-detection.git",
                     "/kaggle/working/project"], check=True)
    print("Project cloned.")
else:
    subprocess.run(["git", "-C", "/kaggle/working/project", "pull", "--ff-only"],
                   check=True)
    print("Project updated.")

# Step 3: Find raw video dataset
print("\n=== Searching for raw video dataset ===")
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}
data_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'real' in dirs or 'fake' in dirs or 'original' in dirs or 'manipulated' in dirs:
        # Check if this directory contains video files (not just images)
        has_videos = False
        for subdir_name in ['real', 'fake', 'original', 'manipulated']:
            subdir = os.path.join(root, subdir_name)
            if os.path.isdir(subdir):
                for sr, sd, sf in os.walk(subdir):
                    for f in sf[:10]:  # Check first 10 files
                        if os.path.splitext(f)[1].lower() in VIDEO_EXTS:
                            has_videos = True
                            break
                    if has_videos:
                        break
            if has_videos:
                break
        if has_videos:
            data_path = root
            break

if data_path is None:
    print("ERROR: No raw video dataset found! Listing /kaggle/input/:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            vids = len([f for f in files if os.path.splitext(f)[1].lower() in VIDEO_EXTS])
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files, {vids} videos)')
    raise RuntimeError("No raw video dataset found!")

print(f"Found raw dataset at: {data_path}")

# Count videos
total_vids = 0
for sr, sd, sf in os.walk(data_path):
    total_vids += sum(1 for f in sf if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
print(f"Total video files: {total_vids}")

# Step 4: Run preprocessing
output_name = f"preprocessed_{DATASET_NAME}_{NUM_FRAMES}"
output_path = f"/kaggle/working/{output_name}"

print(f"\n=== Starting preprocessing ===")
print(f"Input:  {data_path}")
print(f"Output: {output_path}")
print(f"Frames: {NUM_FRAMES}")
print(f"Size:   {OUTPUT_SIZE}x{OUTPUT_SIZE}")

cmd = [
    sys.executable, '/kaggle/working/project/preprocess_videos.py',
    data_path, output_path,
    '--max-frames', str(NUM_FRAMES),
    '--output-size', str(OUTPUT_SIZE),
    '--min-face-confidence', str(MIN_FACE_CONF),
    '--min-detection-ratio', str(MIN_DETECTION_RATIO),
    '--min-saved-faces', str(NUM_FRAMES),
    '--device', 'cuda' if subprocess.run(['python', '-c', 'import torch; print(torch.cuda.is_available())'],
                                          capture_output=True, text=True).stdout.strip() == 'True' else 'cpu',
]

print(f"Command: {' '.join(cmd)}")
result = subprocess.run(cmd, cwd='/kaggle/working/project',
                        env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'})
print(f"\nPreprocessing exited with code: {result.returncode}")

# Step 5: Show summary
import json
summary_path = os.path.join(output_path, 'summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"\n=== PREPROCESSING SUMMARY ===")
    print(f"Total videos processed: {summary['total_videos']}")
    print(f"Saved:   {summary['saved_videos']} (real={summary['real_saved']}, fake={summary['fake_saved']})")
    print(f"Dropped: {summary['dropped_videos']} (real={summary['real_dropped']}, fake={summary['fake_dropped']})")
    print(f"Errors:  {summary['error_videos']}")
    print(f"Mean detection ratio: {summary['mean_detection_ratio_saved']:.4f}")
    if summary.get('drop_reasons'):
        print(f"Drop reasons: {summary['drop_reasons']}")
else:
    print("WARNING: summary.json not found!")

# Step 6: Package for download
print(f"\n=== Packaging output ===")
os.system(f"tar czf /kaggle/working/{output_name}.tar.gz -C /kaggle/working {output_name}/")
tar_size = os.path.getsize(f"/kaggle/working/{output_name}.tar.gz") / (1024**2)
print(f"{output_name}.tar.gz: {tar_size:.1f} MB")
print("Ready for download!")